# unlearn-quant on Kaggle (T4)
Settings: **GPU T4**, **Internet ON**. The harness reaches this notebook either by (a) git clone of your repo, or (b) attaching the `unlearn-quant` folder as a Kaggle Dataset.

Phase A = replication (bnb only). Phase B = edge int4 + distillation.

In [ ]:
# 1. Get the harness into /kaggle/working
import os, shutil, subprocess
REPO_URL = os.environ.get('UNLEARN_QUANT_REPO', 'https://github.com/Arya-126/unlearn-quant.git')  # override via env, or blank to use an attached dataset
WORK = '/kaggle/working/unlearn-quant'
if REPO_URL:
    subprocess.run(['git','clone',REPO_URL,WORK], check=True)
elif os.path.isdir('/kaggle/input/unlearn-quant'):
    shutil.copytree('/kaggle/input/unlearn-quant', WORK, dirs_exist_ok=True)
else:
    raise SystemExit('Provide UNLEARN_QUANT_REPO or attach the unlearn-quant dataset.')
os.chdir(WORK)
print('cwd', os.getcwd(), os.listdir())

In [ ]:
# 2. Core deps (bnb/gptq/awq). GGUF deps installed separately in cell 3.
!pip -q install -r requirements.txt
!pip -q install gptqmodel autoawq || echo 'gptq/awq install issues -> those conditions will be skipped'

In [ ]:
# 3. (Optional) GGUF path: build llama.cpp + install llama-cpp-python. Skip if you don't need GGUF conditions.
import os, subprocess
LCPP = '/kaggle/working/llama.cpp'
if not os.path.isdir(LCPP):
    subprocess.run(['git','clone','--depth','1','https://github.com/ggerganov/llama.cpp',LCPP], check=True)
    subprocess.run(['cmake','-B',f'{LCPP}/build',LCPP], check=True)
    subprocess.run(['cmake','--build',f'{LCPP}/build','--config','Release','-j','2','--target','llama-quantize'], check=True)
    subprocess.run(['pip','-q','install','-r',f'{LCPP}/requirements.txt'], check=True)
!pip -q install llama-cpp-python || echo 'llama-cpp-python install failed -> GGUF conditions will be skipped'
os.environ['LLAMACPP_DIR'] = LCPP

In [ ]:
# 4a. Phase A — replication (run one method per session to stay under the GPU limit)
!python -m scripts.run_replication --method graddiff --n-eval 100

In [ ]:
# 4b. Phase B — edge int4 + distillation (after Phase A passes the recovery gate)
!python -m scripts.run_edge_extension --method graddiff --n-eval 100 --distill

In [ ]:
# 5. Surface results (also persisted to /kaggle/working for download)
import pandas as pd
df = pd.read_csv('results/summary.csv')
import shutil; shutil.copytree('results','/kaggle/working/results', dirs_exist_ok=True)
df[['method','condition','forget_score','recovery_ratio','model_utility','forget_quality']]